# Extensão do estudo | SQL, modelo de adesão e Power BI

Continuação independente da análise exploratória, utilizando **a mesma base UCI Bank Marketing**. Esta etapa consulta indicadores via SQL, testa um modelo simples de propensão à adesão e prepara arquivos para visualização no Power BI.

> Este notebook é complementar ao `01_analise_exploratoria.ipynb`. Não altera a análise original. Execute as células em ordem após enviar `bank-additional-full.csv` para a sessão do Colab.

## Preparação da base

Carrego a mesma base utilizada na EDA e aplico a mesma decisão de remover linhas integralmente repetidas, mantendo a comparação de indicadores consistente. Como não há identificador único de contato, essa decisão tem limitações.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

caminho = '/content/bank-additional-full.csv'
df = pd.read_csv(caminho, sep=';').drop_duplicates().copy()
df['adesao'] = (df['y'] == 'yes').astype(int)

print(f'Registros considerados: {len(df):,}')
print(f'Taxa de adesão: {df["adesao"].mean():.2%}')

## 1. Consultas SQL | Indicadores comerciais

Montei uma tabela temporária no SQLite para consultar a base por SQL, sem configurar um servidor PostgreSQL. A primeira consulta estabelece a taxa geral; as seguintes comparam volume e conversão por profissão e por mês de contato. Os meses representam categorias da campanha, não uma série cronológica completa.

In [ ]:
conexao = sqlite3.connect(':memory:')
df.to_sql('campanhas', conexao, index=False, if_exists='replace')

consulta_profissoes = '''
SELECT job AS profissao,
       COUNT(*) AS contatos,
       SUM(adesao) AS adesoes,
       ROUND(100.0 * AVG(adesao), 2) AS taxa_adesao_pct
FROM campanhas
GROUP BY job
ORDER BY taxa_adesao_pct DESC;
'''

indicadores_profissao_sql = pd.read_sql_query(consulta_profissoes, conexao)
display(indicadores_profissao_sql)

In [ ]:
consulta_meses = '''
SELECT month AS mes,
       COUNT(*) AS contatos,
       SUM(adesao) AS adesoes,
       ROUND(100.0 * AVG(adesao), 2) AS taxa_adesao_pct
FROM campanhas
GROUP BY month
ORDER BY contatos DESC;
'''

indicadores_mes_sql = pd.read_sql_query(consulta_meses, conexao)
display(indicadores_mes_sql)

## 2. Machine Learning | Previsão de adesão

O objetivo agora é testar se informações conhecidas no momento do contato ajudam a distinguir registros com e sem adesão. Escolhi a **Regressão Logística** como modelo inicial por ser uma referência simples e interpretável. Excluí `duration` porque a duração da ligação só é conhecida após o contato e provocaria vazamento de informação em uma previsão pré-contato.

A separação treino/teste é aleatória e demonstra o procedimento; para uma aplicação real, seria necessário avaliar validação temporal, identificação de clientes, consentimento, viés e mudanças de comportamento.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numericas = ['age', 'campaign', 'previous']
categoricas = ['job', 'marital', 'education', 'default', 'housing',
              'loan', 'contact', 'month', 'day_of_week', 'poutcome']
atributos = numericas + categoricas

X = df[atributos]
y = df['adesao']
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preparacao = ColumnTransformer([
    ('num', Pipeline([
        ('imputar', SimpleImputer(strategy='median')),
        ('padronizar', StandardScaler())
    ]), numericas),
    ('cat', Pipeline([
        ('imputar', SimpleImputer(strategy='most_frequent')),
        ('codificar', OneHotEncoder(handle_unknown='ignore'))
    ]), categoricas)
])

modelo = Pipeline([
    ('preparacao', preparacao),
    ('classificador', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
modelo.fit(X_treino, y_treino)
probabilidades = modelo.predict_proba(X_teste)[:, 1]
previsoes = modelo.predict(X_teste)

print(f'ROC-AUC: {roc_auc_score(y_teste, probabilidades):.3f}')
print(f'PR-AUC: {average_precision_score(y_teste, probabilidades):.3f}')
print(classification_report(y_teste, previsoes, zero_division=0))

### Interpretação do modelo

ROC-AUC mede a capacidade de diferenciar adesões e não adesões; PR-AUC é útil porque as adesões são menos frequentes. Essas métricas só devem ser apresentadas **depois de executar a célula**. O modelo é uma demonstração acadêmica, não uma ferramenta de decisão bancária em produção.

## 3. Preparação para Power BI

Para visualizar o desempenho comercial, exportei uma base tratada e dois resumos. O painel poderá comparar o volume de contatos, as adesões e a taxa por profissão e por mês. **Isso prepara os dados; não cria um arquivo `.pbix` automaticamente.**

In [ ]:
dados_powerbi = df[['age', 'job', 'marital', 'education', 'contact',
                     'month', 'day_of_week', 'campaign', 'y', 'adesao']].copy()
dados_powerbi['faixa_etaria'] = pd.cut(
    dados_powerbi['age'],
    bins=[0, 29, 39, 49, 59, 69, float('inf')],
    labels=['Até 29', '30-39', '40-49', '50-59', '60-69', '70+']
)

dados_powerbi.to_csv('/content/campanhas_tratadas.csv', index=False)
indicadores_profissao_sql.to_csv('/content/indicadores_profissao.csv', index=False)
indicadores_mes_sql.to_csv('/content/indicadores_mes.csv', index=False)

print('Arquivos preparados: campanhas_tratadas.csv, indicadores_profissao.csv e indicadores_mes.csv')

In [ ]:
from google.colab import files

# Execute os downloads quando quiser montar o painel no Power BI Desktop.
for nome in ['campanhas_tratadas.csv', 'indicadores_profissao.csv', 'indicadores_mes.csv']:
    files.download('/content/' + nome)

## Próximo passo

No Power BI Desktop, importe `campanhas_tratadas.csv` e crie três cartões: total de contatos, adesões e taxa de adesão. Em seguida, monte barras de taxa por profissão e por faixa etária. A documentação `powerbi/INSTRUCOES.md` traz sugestões de medidas DAX.

**Limites:** a base é histórica e não representa clientes Itaú; o SQL foi escrito para SQLite, não testado em PostgreSQL; o modelo e o dashboard só podem ser considerados concluídos após suas respectivas execuções e validações.